# Solución Big Data – ModaEstilo
Datos: `modaestilo_hechos.csv` (ventas y alquileres) y `modaestilo_devoluciones.csv`, exportados del OLAP **ModaEstilo_DW**.

## Parte 1. Esquema y volumen

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.moda")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.moda.datos")
ruta = "/Volumes/workspace/moda/datos"
display(dbutils.fs.ls(ruta))     # aquí deben estar los 2 CSV

## Parte 2. Lectura de los CSV en DataFrames de Spark (capa Bronze)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DateType, DecimalType

# Esquema declarado (mejora M-07, resultado de EXP-12): importes en decimal exacto, fechas como date e ids como long
esquema_hechos = StructType([
    StructField("tipo_operacion", StringType(), True),
    StructField("id_operacion", LongType(), True),
    StructField("id_detalle", LongType(), True),
    StructField("nro_documento", StringType(), True),
    StructField("fuente", StringType(), True),
    StructField("fecha", DateType(), True),
    StructField("id_cliente", IntegerType(), True),
    StructField("distrito", StringType(), True),
    StructField("id_trabajador", IntegerType(), True),
    StructField("cargo", StringType(), True),
    StructField("id_articulo", IntegerType(), True),
    StructField("articulo", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("canal", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio_unitario", DecimalType(12, 2), True),
    StructField("importe", DecimalType(12, 2), True),
    StructField("estado", StringType(), True),
])

esquema_devol = StructType([
    StructField("id_devolucion", IntegerType(), True),
    StructField("alquiler", StringType(), True),
    StructField("fecha_devolucion", DateType(), True),
    StructField("articulo", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("tipo_falla", StringType(), True),
    StructField("gravedad", StringType(), True),
    StructField("accion", StringType(), True),
    StructField("tipo_devolucion", StringType(), True),
    StructField("dias_retraso", IntegerType(), True),
    StructField("monto_cargo", DecimalType(12, 2), True),
])

dfHechos = (spark.read
  .format("csv")
  .option("header", "true")
  .schema(esquema_hechos)
  .load(f"{ruta}/modaestilo_hechos.csv"))

dfDevol = (spark.read
  .format("csv")
  .option("header", "true")
  .schema(esquema_devol)
  .load(f"{ruta}/modaestilo_devoluciones.csv"))

# overwriteSchema: reemplaza las tablas creadas antes con inferSchema (tenían importes double)
dfHechos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.bronze_hechos")
dfDevol.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.bronze_devoluciones")

In [ ]:
dfHechos.printSchema()
dfDevol.printSchema()

In [ ]:
display(dfHechos)
print("Filas de hechos:", dfHechos.count(), "| Filas de devoluciones:", dfDevol.count())

## Parte 3. Análisis exploratorio (EDA)

In [ ]:
from pyspark.sql import functions as F

# Estadísticas básicas
display(dfHechos.describe("cantidad", "precio_unitario", "importe"))

In [ ]:
# Valores nulos por columna
display(dfHechos.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in dfHechos.columns]))

In [ ]:
# Operaciones, unidades e importe por tipo
display(dfHechos.groupBy("tipo_operacion")
        .agg(F.countDistinct("id_operacion").alias("operaciones"),
             F.sum("cantidad").alias("unidades"),
             F.round(F.sum("importe"), 2).alias("importe")))

In [ ]:
# Importe por categoría y por canal
display(dfHechos.groupBy("categoria").agg(F.round(F.sum("importe"), 2).alias("importe")).orderBy(F.desc("importe")))
display(dfHechos.groupBy("canal").count().orderBy(F.desc("count")))

In [ ]:
# Valores distintos y posibles duplicados
display(dfHechos.select("estado").distinct())
print("Filas:", dfHechos.count(), "| Filas distintas:", dfHechos.distinct().count())

In [ ]:
# Muestra aproximada del 10 % (con semilla para repetir el resultado)
muestra = dfHechos.sample(0.10, seed=42)
print("Filas en la muestra:", muestra.count())
display(muestra)

In [ ]:
# Devoluciones: tipo de devolución y tipo de falla
display(dfDevol.groupBy("tipo_devolucion").count().orderBy(F.desc("count")))
display(dfDevol.filter(F.col("tipo_falla") != "Ninguna")
        .groupBy("tipo_falla", "gravedad").agg(F.count("*").alias("prendas"), F.sum("monto_cargo").alias("cargos")))

## Parte 4. Limpieza y preparación (capa Silver)

In [ ]:
dfHechos2 = (spark.table("workspace.moda.bronze_hechos")
  .withColumnRenamed("precio_unitario", "precio")
  .withColumnRenamed("importe", "monto")
  .select("tipo_operacion", "id_operacion", "id_detalle", "nro_documento", "fuente", "fecha",
          "distrito", "cargo", "articulo", "categoria", "canal", "cantidad", "precio", "monto", "estado")
  .withColumn("fecha", F.to_date("fecha"))
  .withColumn("monto", F.col("monto").cast("decimal(12,2)"))
  .withColumn("precio", F.col("precio").cast("decimal(12,2)"))
  .na.drop(subset=["id_detalle", "fecha", "monto"])
  .na.fill({"distrito": "No registrado", "canal": "No registrado"})
  .filter((F.col("cantidad") > 0) & (F.col("monto") > 0))
  .dropDuplicates(["tipo_operacion", "id_detalle"])
  .withColumn("anio", F.year("fecha"))
  .withColumn("mes", F.month("fecha"))
  .withColumn("anio_mes", F.date_format("fecha", "yyyy-MM")))

display(dfHechos2)

In [ ]:
# Join: a cada alquiler se le agrega su devolución (tipo de devolución, falla y cargo)
dfDevol2 = (spark.table("workspace.moda.bronze_devoluciones")
  .withColumnRenamed("monto_cargo", "cargo_devolucion")
  .select("alquiler", "articulo", "fecha_devolucion", "tipo_devolucion", "tipo_falla", "gravedad", "cargo_devolucion"))

dfAlquileres = (dfHechos2.filter("tipo_operacion = 'ALQUILER'")
  .withColumn("alquiler", F.concat_ws("-", "fuente", "nro_documento"))
  .join(dfDevol2, on=["alquiler", "articulo"], how="left")
  .na.fill({"tipo_devolucion": "Sin devolver", "tipo_falla": "Sin devolver"}))

display(dfAlquileres.selectExpr("alquiler", "articulo", "estado", "tipo_devolucion", "tipo_falla", "cargo_devolucion"))

In [ ]:
# Comprobación: los totales deben cuadrar con ModaEstilo_DW (ventas S/ 38 056.40 y alquileres S/ 14 118.00)
totales = {r["tipo_operacion"]: float(r["total"])
           for r in dfHechos2.groupBy("tipo_operacion").agg(F.sum("monto").alias("total")).collect()}
print(totales)
assert abs(totales["VENTA"] - 38056.40) < 0.01 and abs(totales["ALQUILER"] - 14118.00) < 0.01
print("Silver cuadra con el OLAP")

dfHechos2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.silver_hechos")
dfAlquileres.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.silver_alquileres")

## Parte 5. Indicadores (capa Gold)

In [ ]:
gold_mes = (dfHechos2.groupBy("anio_mes", "tipo_operacion")
  .agg(F.sum("monto").alias("ingreso"),
       F.countDistinct("id_operacion").alias("operaciones"),
       F.sum("cantidad").alias("unidades"))
  .withColumn("ticket_promedio", F.round(F.col("ingreso") / F.col("operaciones"), 2))
  .orderBy("anio_mes"))

gold_categoria = (dfHechos2.groupBy("categoria", "tipo_operacion")
  .agg(F.sum("monto").alias("ingreso"), F.sum("cantidad").alias("unidades")))

gold_canal = (dfHechos2.groupBy("canal")
  .agg(F.sum("monto").alias("ingreso"), F.countDistinct("id_operacion").alias("operaciones")))

gold_devoluciones = (dfAlquileres.groupBy("tipo_devolucion", "tipo_falla")
  .agg(F.count("*").alias("prendas"), F.sum("cargo_devolucion").alias("cargos")))

gold_mes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.gold_ingresos_mes")
gold_categoria.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.gold_ingresos_categoria")
gold_canal.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.gold_ingresos_canal")
gold_devoluciones.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.gold_devoluciones")

## Parte 6. Visualizaciones (en cada resultado: + > Visualization)

In [ ]:
%sql
-- Ingresos por mes (gráfico de líneas: X anio_mes, Y ingreso, agrupar por tipo_operacion)
SELECT anio_mes, tipo_operacion, ingreso, operaciones, ticket_promedio
FROM workspace.moda.gold_ingresos_mes
ORDER BY anio_mes

In [ ]:
%sql
-- Ingresos por categoría (barras)
SELECT categoria, tipo_operacion, ingreso, unidades
FROM workspace.moda.gold_ingresos_categoria
ORDER BY ingreso DESC

In [ ]:
%sql
-- Ingresos por canal (barras o anillo)
SELECT canal, ingreso, operaciones
FROM workspace.moda.gold_ingresos_canal
ORDER BY ingreso DESC

In [ ]:
%sql
-- Devoluciones por tipo y falla (barras apiladas)
SELECT tipo_devolucion, tipo_falla, prendas, cargos
FROM workspace.moda.gold_devoluciones
ORDER BY prendas DESC